# Assignment 4 — ActiveLearningAgent

Этот ноутбук показывает, как выбирать следующие примеры для ручной разметки после первого раунда annotation / HITL.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "agents").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from sklearn.model_selection import train_test_split

from agents.active_learning_agent import ActiveLearningAgent

candidate_paths = [
    ROOT / "data/labeled/final_dataset.parquet",
    ROOT / "data/raw/merged_raw.csv",
]
for path in candidate_paths:
    if path.exists():
        df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
        break
else:
    raise FileNotFoundError("No input dataset found in data/labeled/final_dataset.parquet or data/raw/merged_raw.csv")

if "final_label" in df.columns and "label" not in df.columns:
    df = df.rename(columns={"final_label": "label"})

agent = ActiveLearningAgent(config=ROOT / "config.yaml")
df = df.dropna(subset=["text", "label"]).copy()
labeled_df, pool_df = train_test_split(df, train_size=min(50, max(1, len(df) - 1)), random_state=42)
batch = agent.select_batch(pool_df=pool_df, labeled_df=labeled_df, strategy="entropy", batch_size=min(20, len(pool_df)))
batch[[c for c in ["text", "predicted_label", "uncertainty", "margin"] if c in batch.columns]].head()

In [ ]:
histories = {}
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
start_n = min(50, max(2, len(train_df) // 3))
labeled_seed, pool_seed = train_test_split(train_df, train_size=start_n, random_state=42)

for strategy in ["entropy", "random"]:
    histories[strategy] = agent.run_cycle(
        labeled_df=labeled_seed.copy(),
        pool_df=pool_seed.copy(),
        test_df=test_df.copy(),
        strategy=strategy,
        n_iterations=3,
        batch_size=min(20, len(pool_seed)),
    )

histories

In [ ]:
import matplotlib.pyplot as plt

history_df = pd.concat(
    [pd.DataFrame(rows).assign(strategy=name) for name, rows in histories.items()],
    ignore_index=True,
)
history_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for strategy_name, sub in history_df.groupby("strategy"):
    ax.plot(sub["iteration"], sub["f1_macro"], marker="o", label=strategy_name)
ax.set_title("Active learning comparison")
ax.set_xlabel("iteration")
ax.set_ylabel("f1_macro")
ax.legend()
plt.tight_layout()
plt.show()

## Что показать на защите

- что `entropy` выбирает более сомнительные примеры, чем `random`;
- что батч содержит `predicted_label`, `uncertainty` и `margin`;
- почему именно такие примеры полезно отправлять человеку для следующего раунда разметки.